# 07 — Tree of Thoughts

**Model:** DeepSeek-V3 via Nebius AI Studio  
**Pattern:** Parallel Reasoning Branches with Scoring & Pruning  
**Difficulty:** ⭐⭐⭐⭐⭐

---

## The Problem

Standard chain-of-thought reasoning is linear — one thought follows the previous. If the model takes a wrong turn early, the entire chain is compromised with no way to backtrack.

**Tree of Thoughts (ToT)** (Yao et al., 2023) treats reasoning as a *search problem*: the model generates multiple candidate reasoning paths from each state, scores them, and only expands the most promising branches — discarding weak paths early. This mirrors how humans approach hard problems: we consider alternatives before committing.

## Algorithm

```
Root Problem
     │
     ├── Thought A ──── Score: 8/10 ── expand
     │        ├── A1 ── Score: 9/10 ── BEST PATH
     │        └── A2 ── Score: 5/10 ── prune
     ├── Thought B ──── Score: 6/10 ── expand
     │        └── B1 ── Score: 4/10 ── prune
     └── Thought C ──── Score: 3/10 ── prune (never expanded)
```

## Architecture (LangGraph implementation)

```
┌──────────────────────────────────────────────────────────┐
│                   Tree of Thoughts Graph                  │
│                                                           │
│  START → [Generate Thoughts] → [Score Thoughts]          │
│               ↑                      │                   │
│               │               [Prune & Select]           │
│               │                      │                   │
│               └─── more depth ───────┘                   │
│                          │                               │
│                     depth limit                          │
│                          │                               │
│                    [Synthesize] → END                    │
└──────────────────────────────────────────────────────────┘
```

## Setup

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_nebius import ChatNebius
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Optional
from pydantic import BaseModel
import json

llm = ChatNebius(
    model="deepseek-ai/DeepSeek-V3",
    temperature=0.7,  # Higher temp for thought diversity
    api_key=os.environ["NEBIUS_API_KEY"]
)

# Separate evaluator with lower temperature for consistent scoring
evaluator_llm = ChatNebius(
    model="deepseek-ai/DeepSeek-V3",
    temperature=0,
    api_key=os.environ["NEBIUS_API_KEY"]
)

print("Setup complete.")

## Data Structures

In [ ]:
class Thought(BaseModel):
    content: str
    score: float = 0.0        # 0–10 rating assigned by evaluator
    depth: int = 0            # Which level of the tree this is at
    parent_content: str = "" # The parent thought this branched from


class ToTState(TypedDict):
    problem: str
    active_thoughts: List[Thought]    # Current frontier (thoughts being evaluated)
    pruned_thoughts: List[Thought]    # Thoughts that scored below threshold
    best_path: List[Thought]          # Top-scoring path across all depths
    current_depth: int
    max_depth: int
    branching_factor: int             # How many thoughts to generate per parent
    prune_threshold: float            # Score below this gets pruned
    final_answer: Optional[str]

## Node: Generate Thoughts

For each active thought (or the root problem at depth 0), generate `branching_factor` candidate next thoughts.

In [ ]:
THOUGHT_GEN_PROMPT = """You are exploring different reasoning approaches for a problem.
Given the problem and a current reasoning step, generate {n} DISTINCT next thoughts.
Each thought should explore a meaningfully different angle or approach.

Return ONLY a JSON array of strings: ["thought 1", "thought 2", ...]
"""


def generate_thoughts_node(state: ToTState) -> dict:
    """Expand the frontier by generating new thoughts from each active thought."""
    print(f"\n[Generate] Depth {state['current_depth']} — generating thoughts...")

    new_thoughts = []

    if state["current_depth"] == 0:
        # Root: generate initial thoughts directly from problem
        parents = [{"content": state["problem"], "parent_content": ""}]
    else:
        parents = state["active_thoughts"]

    for parent in parents:
        parent_text = parent["content"] if isinstance(parent, dict) else parent.content
        context = f"Problem: {state['problem']}\n\nCurrent reasoning: {parent_text}"

        response = llm.invoke([
            SystemMessage(content=THOUGHT_GEN_PROMPT.format(n=state["branching_factor"])),
            HumanMessage(content=context)
        ])

        raw = response.content.strip()
        json_str = raw[raw.find("["):raw.rfind("]")+1]
        try:
            thought_texts = json.loads(json_str)
        except Exception:
            thought_texts = [line.strip() for line in raw.split("\n") if line.strip()]

        for text in thought_texts[:state["branching_factor"]]:
            new_thoughts.append(Thought(
                content=text,
                depth=state["current_depth"],
                parent_content=parent_text
            ))
        print(f"  Generated {len(thought_texts)} thoughts from parent: '{parent_text[:60]}...'")

    return {"active_thoughts": new_thoughts}

## Node: Score Thoughts

Each thought is evaluated by the evaluator LLM on a 0–10 scale.

In [ ]:
SCORER_PROMPT = """You are evaluating a reasoning step for solving a problem.
Score this thought from 0 to 10 based on:
- Relevance to the problem (does it move toward a solution?)
- Logical soundness (is the reasoning valid?)
- Novelty (does it add something new, not just restate the problem?)

Return ONLY a single number (e.g., 7.5). Nothing else.
"""


def score_thoughts_node(state: ToTState) -> dict:
    """Assign a quality score to each active thought."""
    print(f"\n[Score] Evaluating {len(state['active_thoughts'])} thoughts...")

    scored = []
    for thought in state["active_thoughts"]:
        response = evaluator_llm.invoke([
            SystemMessage(content=SCORER_PROMPT),
            HumanMessage(content=f"Problem: {state['problem']}\n\nThought to score: {thought.content}")
        ])
        try:
            score = float(response.content.strip())
        except ValueError:
            score = 5.0  # default if parsing fails

        thought.score = score
        scored.append(thought)
        print(f"  Score {score:.1f}/10 — '{thought.content[:70]}...'")

    # Sort by score descending
    scored.sort(key=lambda t: t.score, reverse=True)
    return {"active_thoughts": scored}

## Node: Prune and Route

In [ ]:
def prune_node(state: ToTState) -> dict:
    """Keep only thoughts above the threshold; track the current best path."""
    surviving = [t for t in state["active_thoughts"] if t.score >= state["prune_threshold"]]
    pruned = [t for t in state["active_thoughts"] if t.score < state["prune_threshold"]]

    # Keep at most top-3 to control branching
    surviving = surviving[:3]

    print(f"\n[Prune] Kept {len(surviving)}, pruned {len(pruned)} thoughts.")

    # Track best path: the chain from root to highest scoring leaf so far
    best_path = state["best_path"].copy() if state["best_path"] else []
    if surviving:
        best_path.append(surviving[0])  # top-scoring thought at this depth

    return {
        "active_thoughts": surviving,
        "pruned_thoughts": state["pruned_thoughts"] + pruned,
        "best_path": best_path,
        "current_depth": state["current_depth"] + 1
    }


def should_expand_or_finish(state: ToTState) -> str:
    if not state["active_thoughts"]:
        print("\n[Router] All branches pruned — synthesizing from best path.")
        return "synthesize"
    if state["current_depth"] >= state["max_depth"]:
        print(f"\n[Router] Max depth {state['max_depth']} reached — synthesizing.")
        return "synthesize"
    return "generate"

## Node: Synthesize Final Answer

In [ ]:
def synthesize_node(state: ToTState) -> dict:
    """Combine the best reasoning path into a final coherent answer."""
    print("\n[Synthesize] Building final answer from best reasoning path...")

    path_text = "\n".join(
        f"Depth {t.depth}, Score {t.score:.1f}: {t.content}"
        for t in state["best_path"]
    )

    response = llm.invoke([
        SystemMessage(content="You synthesize a chain of reasoning steps into a clear, complete final answer."),
        HumanMessage(content=(
            f"Problem: {state['problem']}\n\n"
            f"Best reasoning path explored:\n{path_text}\n\n"
            "Write a clear, direct final answer:"
        ))
    ])

    return {"final_answer": response.content}

## Build the Graph

In [ ]:
builder = StateGraph(ToTState)
builder.add_node("generate", generate_thoughts_node)
builder.add_node("score", score_thoughts_node)
builder.add_node("prune", prune_node)
builder.add_node("synthesize", synthesize_node)

builder.set_entry_point("generate")
builder.add_edge("generate", "score")
builder.add_edge("score", "prune")
builder.add_conditional_edges(
    "prune",
    should_expand_or_finish,
    {"generate": "generate", "synthesize": "synthesize"}
)
builder.add_edge("synthesize", END)

graph = builder.compile()
print("Tree of Thoughts graph compiled.")

## Live Demo

**Scenario:** A strategic problem with multiple valid approaches — ideal for ToT since there's no single obvious path.

In [ ]:
problem = (
    "A startup has $50,000 in remaining runway and 90 days to either raise a new round "
    "or reach profitability. They have a SaaS product with 200 free users and 5 paying customers "
    "at $99/month. What should they prioritize?"
)

initial = {
    "problem": problem,
    "active_thoughts": [],
    "pruned_thoughts": [],
    "best_path": [],
    "current_depth": 0,
    "max_depth": 2,
    "branching_factor": 3,
    "prune_threshold": 6.0,
    "final_answer": None
}

result = graph.invoke(initial)

In [ ]:
print("\n" + "="*60)
print("BEST REASONING PATH")
print("="*60)
for thought in result["best_path"]:
    print(f"[Depth {thought.depth}, Score {thought.score:.1f}] {thought.content}")

print("\n" + "="*60)
print("FINAL ANSWER")
print("="*60)
print(result["final_answer"])

## Key Takeaways

| Concept | Implementation |
|---------|---------------|
| **Breadth-first exploration** | Generate multiple thoughts per depth level |
| **Beam search** | Keep only top-k thoughts, discard the rest |
| **Separate generator/evaluator** | Different temperature settings for creativity vs. reliability |
| **Cost control** | `max_depth` × `branching_factor` bounds total LLM calls |

## What's Next

**Notebook 08** addresses scale — when you have a large amount of input data, **Map-Reduce** lets you process it in parallel and synthesize a coherent output.